# Project 1 — Heart Disease Prediction
Binary classification on the BRFSS 2015 dataset (`_MICHD`).

**Pipeline:**
1. Load clean data
2. Impute NaNs (median, fit on train only)
3. Standardize features (fit on train only)
4. Handle class imbalance (oversampling + threshold tuning)
5. Train with regularized logistic regression
6. Evaluate with cross-validation (AUC + F1)
7. Generate submission

In [1]:
import numpy as np
from helpers import create_csv_submission
from implementations import (
    logistic_regression,
    reg_logistic_regression,
    ROC_AUC,
)

## 1. Load Data

In [2]:
CLEAN_DIR = "data/clean/"

x_train = np.load(CLEAN_DIR + "x_train_clean.npy")
x_test  = np.load(CLEAN_DIR + "x_test_clean.npy")
y_train = np.load(CLEAN_DIR + "y_train.npy")       # labels: -1 / 1
train_ids = np.load(CLEAN_DIR + "train_ids.npy")
test_ids  = np.load(CLEAN_DIR + "test_ids.npy")

n_pos = (y_train == 1).sum()
n_neg = (y_train == -1).sum()
print("x_train:", x_train.shape)
print("x_test :", x_test.shape)
print(f"Labels — +1: {n_pos}  -1: {n_neg}  imbalance ratio: {n_neg/n_pos:.1f}x")

x_train: (328135, 275)
x_test : (109379, 275)
Labels — +1: 28975  -1: 299160  imbalance ratio: 10.3x


## 2. Preprocessing helpers

In [3]:
def drop_high_nan_cols(x_tr, x_te, threshold=0.8):
    """Drop columns where the training NaN rate exceeds `threshold`."""
    nan_rate = np.isnan(x_tr).mean(axis=0)
    keep = nan_rate < threshold
    return x_tr[:, keep], x_te[:, keep], keep

def impute_median(x_tr, x_te):
    """Replace NaNs with the training-set median (fit on train only)."""
    medians = np.nanmedian(x_tr, axis=0)
    def fill(x):
        out = x.copy()
        for j in range(x.shape[1]):
            mask = np.isnan(out[:, j])
            out[mask, j] = medians[j]
        return out
    return fill(x_tr), fill(x_te)

def standardize(x_tr, x_te):
    """Zero-mean unit-variance scaling (fit on train only)."""
    mu  = x_tr.mean(axis=0)
    std = x_tr.std(axis=0)
    std[std == 0] = 1.0
    return (x_tr - mu) / std, (x_te - mu) / std

def add_bias(x):
    """Prepend a column of ones (intercept term)."""
    return np.c_[np.ones(x.shape[0]), x]

def oversample_positives(x, y, seed=42):
    """Duplicate positive samples until classes are balanced."""
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_repeat = len(neg_idx) // len(pos_idx)
    remainder = len(neg_idx) % len(pos_idx)
    extra = rng.choice(pos_idx, remainder, replace=False)
    idx = np.concatenate([neg_idx, np.tile(pos_idx, n_repeat), extra])
    idx = rng.permutation(idx)
    return x[idx], y[idx]

def f1_score(y_true, y_pred):
    """F1 score for binary labels (0/1)."""
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall    = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)

def best_threshold(y_true, scores, thresholds=np.arange(0.1, 0.9, 0.02)):
    """Find the threshold that maximises F1 on the given set."""
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_true, (scores >= t).astype(float))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

## 3. Build feature matrix

In [4]:
NAN_THRESHOLD = 0.8

x_tr, x_te, kept = drop_high_nan_cols(x_train, x_test, NAN_THRESHOLD)
print(f"Columns kept: {kept.sum()} / {len(kept)}")

x_tr, x_te = impute_median(x_tr, x_te)
x_tr, x_te = standardize(x_tr, x_te)

tx_tr = add_bias(x_tr)
tx_te = add_bias(x_te)

# Labels: -1/1 → 0/1
y01 = (y_train == 1).astype(float)

print("tx_train:", tx_tr.shape)
print("NaN remaining:", np.isnan(tx_tr).sum())

Columns kept: 166 / 275
tx_train: (328135, 167)
NaN remaining: 0


## 4. Cross-validation with threshold tuning

In [5]:
def cross_validate(train_fn, tx, y, k=5, use_oversample=True, seed=42):
    """
    k-fold CV. For each fold:
      - optionally oversample positives in the training fold
      - find the best threshold on the validation fold
    Returns mean AUC, mean F1, mean best-threshold.
    """
    n = len(y)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    folds = np.array_split(idx, k)

    aucs, f1s, thresholds = [], [], []
    for i in range(k):
        val_idx = folds[i]
        tr_idx  = np.concatenate([folds[j] for j in range(k) if j != i])

        tx_fold, y_fold = tx[tr_idx], y[tr_idx]
        if use_oversample:
            tx_fold, y_fold = oversample_positives(tx_fold, y_fold, seed=seed+i)

        w, _ = train_fn(y_fold, tx_fold)

        score = 0.5 * (1 + np.tanh(0.5 * tx[val_idx].dot(w)))
        _, _, auc = ROC_AUC(y[val_idx], score)
        t, f1   = best_threshold(y[val_idx], score)

        aucs.append(auc)
        f1s.append(f1)
        thresholds.append(t)

    print(f"  AUC:       {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"  F1:        {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"  Threshold: {np.mean(thresholds):.2f} ± {np.std(thresholds):.2f}")
    return np.mean(aucs), np.mean(f1s), np.mean(thresholds)

## 5. Logistic Regression baseline

In [6]:
MAX_ITERS = 100
GAMMA     = 0.1

def train_logistic(y, tx):
    return logistic_regression(y, tx, np.zeros(tx.shape[1]), MAX_ITERS, GAMMA)

print("Logistic Regression:")
auc, f1, thr = cross_validate(train_logistic, tx_tr, y01)

Logistic Regression:
  AUC:       0.8570 ± 0.0020
  F1:        0.4213 ± 0.0027
  Threshold: 0.73 ± 0.01


## 6. Regularized Logistic Regression

In [7]:
LAMBDA = 1e-4

def train_reg_logistic(y, tx):
    return reg_logistic_regression(y, tx, LAMBDA, np.zeros(tx.shape[1]), MAX_ITERS, GAMMA)

print("Regularized Logistic Regression:")
auc, f1, thr = cross_validate(train_reg_logistic, tx_tr, y01)

Regularized Logistic Regression:
  AUC:       0.8570 ± 0.0020
  F1:        0.4213 ± 0.0027
  Threshold: 0.73 ± 0.01


## 7. Tune lambda

In [8]:
lambdas = [1e-5, 1e-4, 1e-3, 1e-2]

results = []
for lam in lambdas:
    def train_fn(y, tx, lam=lam):
        return reg_logistic_regression(y, tx, lam, np.zeros(tx.shape[1]), MAX_ITERS, GAMMA)
    print(f"\nlambda={lam:.0e}:")
    auc, f1, thr = cross_validate(train_fn, tx_tr, y01)
    results.append((lam, auc, f1, thr))

best = max(results, key=lambda r: r[2])   # pick best F1
BEST_LAMBDA, BEST_THRESHOLD = best[0], best[3]
print(f"\nBest lambda: {BEST_LAMBDA:.0e}  F1={best[2]:.4f}  threshold={BEST_THRESHOLD:.2f}")


lambda=1e-05:
  AUC:       0.8570 ± 0.0020
  F1:        0.4213 ± 0.0028
  Threshold: 0.73 ± 0.01

lambda=1e-04:
  AUC:       0.8570 ± 0.0020
  F1:        0.4213 ± 0.0027
  Threshold: 0.73 ± 0.01

lambda=1e-03:
  AUC:       0.8569 ± 0.0020
  F1:        0.4211 ± 0.0026
  Threshold: 0.73 ± 0.01

lambda=1e-02:
  AUC:       0.8567 ± 0.0020
  F1:        0.4210 ± 0.0025
  Threshold: 0.74 ± 0.01

Best lambda: 1e-04  F1=0.4213  threshold=0.73


## 8. Train final model on full training set

In [9]:
# Oversample on the full training set
tx_balanced, y_balanced = oversample_positives(tx_tr, y01)
print(f"Balanced training set: {tx_balanced.shape}  pos={y_balanced.sum():.0f}  neg={(y_balanced==0).sum():.0f}")

w_final, final_loss = reg_logistic_regression(
    y_balanced, tx_balanced, BEST_LAMBDA, np.zeros(tx_tr.shape[1]), MAX_ITERS, GAMMA
)

train_score = 0.5 * (1 + np.tanh(0.5 * tx_tr.dot(w_final)))
train_f1    = f1_score(y01, (train_score >= BEST_THRESHOLD).astype(float))
_, _, train_auc = ROC_AUC(y01, train_score)

print(f"Final loss : {final_loss:.4f}")
print(f"Train AUC  : {train_auc:.4f}")
print(f"Train F1   : {train_f1:.4f}  (threshold={BEST_THRESHOLD:.2f})")

Balanced training set: (598320, 167)  pos=299160  neg=299160
Final loss : 0.4748
Train AUC  : 0.8575
Train F1   : 0.4213  (threshold=0.73)


## 9. Diagnostic — confusion matrix

In [10]:
## 10. Generate submission

## 9. Generate submission

In [11]:
test_score = 0.5 * (1 + np.tanh(0.5 * tx_te.dot(w_final)))

y_pred = np.where(test_score >= BEST_THRESHOLD, 1, -1)

print(f"Predicted +1: {(y_pred == 1).sum()}   Predicted -1: {(y_pred == -1).sum()}")
print(f"Positive rate: {(y_pred==1).mean()*100:.1f}%")

create_csv_submission(test_ids, y_pred, "submission.csv")
print("Saved → submission.csv")

Predicted +1: 15828   Predicted -1: 93551
Positive rate: 14.5%
Saved → submission.csv
